# First image in JupyterLab on a GPU (optional)

Run this notebook inside a **Nebius GPU DevLab** ([JupyterLab template](https://docs.nebius.com/serverless/quickstart/devlabs#web-console)). It loads Sana, an open text-to-image model, directly onto the DevLab GPU and runs the exact same examples, consistency experiments, practice scenes and share card as `first_image_local.ipynb`—without creating an Endpoint.

**Workflow:**
1. In the Nebius web console, go to **Serverless AI → Devlabs → Create Devlab**, choose **Template → JupyterLab**, and pick an **L40S** GPU if offered.
2. Open the DevLab's public HTTPS URL in your browser, sign in with your Jupyter token, and open a **Terminal**.
3. In the terminal, update the repo and create the GPU environment (once per DevLab). `uv` is already installed:

```bash
cd ~/work/production-ai-agents-for-media-generation
git pull origin 01-draft

uv sync --extra gpu --extra notebooks
uv run python -m ipykernel install --user --name course-gpu --display-name "course (GPU)"
```
4. In JupyterLab choose **Kernel → Change kernel → course (GPU)**, open `01-consistent-media-generation-on-serverless-gpus/code/first_image_devlab.ipynb`, and run it top to bottom. Do not use the template's default conda kernel: its PyTorch pieces are built for different CUDA versions.
5. Download your run archive in the last step and **delete** the DevLab in the console.

> A DevLab bills its GPU while running, and a stopped DevLab still bills its container disk until you **delete** it. Working from your laptop with an Endpoint instead? Use `first_image_local.ipynb`.

Guide: [02-first-generation.md](../02-first-generation.md). Setup (account, credits): the **Setup** section of the [module README](../README.md).

## 1. Check the GPU environment

Dependencies live in a **uv** virtualenv, created once in a terminal from the repo root. This notebook does not install packages.

```bash
uv sync --extra gpu --extra notebooks
uv run python -m ipykernel install --user --name course-gpu --display-name "course (GPU)"
```

If this cell says you are on the conda kernel, stop, run those commands, then **Kernel → Change kernel → course (GPU)**.

The check confirms CUDA and that Diffusers can import Sana. The venv does not include the template's mismatched `torchaudio`.


In [ ]:
import importlib
import os
import shutil
import subprocess
import sys
from pathlib import Path

import torch
from IPython.display import Image, display

if "/opt/conda" in Path(torch.__file__).as_posix():
    raise RuntimeError(
        "This is the DevLab conda kernel. In a terminal, from the repo root, run:\n"
        "  uv sync --extra gpu --extra notebooks\n"
        '  uv run python -m ipykernel install --user --name course-gpu --display-name "course (GPU)"\n'
        "Then Kernel → Change kernel → course (GPU)."
    )

MODULE = "01-consistent-media-generation-on-serverless-gpus"
candidates = [Path.cwd(), Path.cwd() / "code", Path.cwd() / MODULE / "code"]
found = [p for p in candidates if (p / "common.py").is_file()]
assert found, "Open this notebook from the cloned repository (its code/ folder)"
CODE = found[0].resolve()
sys.path.insert(0, str(CODE))

os.environ.setdefault("HF_HOME", str(Path.home() / ".cache" / "huggingface"))

assert torch.cuda.is_available(), "CUDA is not available. Run this notebook on a GPU DevLab."
import torchvision

print(
    "Python:",
    sys.executable,
    "\nGPU:",
    torch.cuda.get_device_name(0),
    "| PyTorch:",
    torch.__version__,
    "| torchvision:",
    torchvision.__version__,
    "| CUDA:",
    torch.version.cuda,
)
print("HF_HOME:", os.environ["HF_HOME"])
print(
    subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.used,memory.total", "--format=csv"], text=True
    )
)
assert torch.__version__.startswith("2.13"), (
    f"Expected torch 2.13 from the course venv, got {torch.__version__}"
)
assert torchvision.__version__.startswith("0.28"), (
    f"Expected torchvision 0.28 from the course venv, got {torchvision.__version__}"
)
from transformers import Gemma2PreTrainedModel  # noqa: F401
from diffusers import SanaPipeline  # noqa: F401

print("transformers + diffusers import OK")

import image

importlib.reload(image)
from common import asset_record, metrics, output_dir, write_json
from image import ImageGenerator, image_comparison, share_card


## 2. Load the model onto the GPU

Instead of polling a remote Endpoint over HTTP, `ImageGenerator.local()` downloads `Efficient-Large-Model/Sana_1600M_1024px_diffusers` (pinned to a Hub commit) on first run and loads it directly into GPU memory.

Keep this kernel alive while you work: every cell below reuses the loaded pipeline in VRAM without paying the model load time (`load_s`) again.

In [ ]:
generator = ImageGenerator.local()

## 3. Generate an image

With the pipeline in GPU memory, we generate a `512x512` image directly in this Python process. Change the prompt if you like. Keep it free of real people, brands and known characters. Each side of `size` must be a multiple of 32.

Passing `seed=None` leaves the seed unset, so every run of this cell generates a new image.

In [ ]:
prompt = (
    "A storybook illustration of a brass telescope glowing on snow under a starry night sky, "
    "a small decorated tree and wrapped presents, warm colors, no people, no text"
)

preview = generator.generate_image(prompt, out="runs/devlab-preview.png", seed=None, size="512x512")
print(preview)
display(Image(filename=str(preview)))

`generator.generate_image(prompt)` uses a default seed of 7 (`seed=7`), saves the PNG to `runs/first-image.png`, and returns the path so the experiments below can repeat an image. Each call is one line.

In [ ]:
out = generator.generate_image(prompt)
print(out)
display(Image(filename=str(out)))

## 4. Same settings, same image

Generate the same image three times with the exact same settings, then compare the three hashes. With the same prompt, seed and settings on the same GPU, are the three images byte-identical?

This cell also writes `metrics.json` (`load_s`, `latency_s`, `peak_vram_gb`, and `assets`) to your run folder so you can use it in Lesson 05 (`cost_per_asset.ipynb`) and Homework 1.

In [ ]:
import importlib.metadata

folder = output_dir(None)
print(folder)
paths, latencies = [], []
for index in range(3):
    path = generator.generate_image(prompt, out=folder / f"asset-{index:03d}.png", seed=7)
    paths.append(path)
    latencies.append(round(generator.last_s, 4))

hashes = [asset_record(path)["sha256"] for path in paths]
print("Hashes:", [h[:12] for h in hashes], "| identical:", len(set(hashes)) == 1)

m = metrics(
    {
        "model": generator.model,
        "model_revision": generator.model_revision,
        "variant": generator.variant,
        "prompt": prompt,
        "seed": 7,
        "params": {
            "width": 512,
            "height": 512,
            "num_inference_steps": 20,
            "guidance_scale": 4.5,
            "negative_prompt": "",
        },
    },
    mode="jupyterlab",
)
m["load_s"] = round(generator.load_s, 4) if generator.load_s is not None else None
m["latency_s"] = latencies
m["env"].update(
    gpu=torch.cuda.get_device_name(0),
    cuda=torch.version.cuda,
    torch=torch.__version__,
    diffusers=importlib.metadata.version("diffusers"),
    peak_vram_gb=round(generator.peak_vram_gb, 3) if generator.peak_vram_gb is not None else None,
)
m["assets"] = [asset_record(path) for path in paths]
m["status"] = "completed"
write_json(folder / "metrics.json", m)
print(folder / "metrics.json")

display(Image(data=image_comparison(paths)))

## 5. Leave out the seed

Same prompt and settings, but no seed: the pipeline picks a random one for every call. Two calls, two different images. A seed you didn't record is a setting you can't repeat.

In [ ]:
no_seed = []
for i in (1, 2):
    path = generator.generate_image(prompt, out=folder / f"no-seed-{i}.png", seed=None)
    no_seed.append(asset_record(path))

display(Image(data=image_comparison([folder / a["file"] for a in no_seed])))

print("Identical:", no_seed[0]["sha256"] == no_seed[1]["sha256"])

## 6. The seed is a setting

Now a character: the same description goes into every prompt below. First, the same scene with seed 7 and seed 8. The seed changes the picture as much as any other setting does.

`generate_image` below hashes the model, prompt and seed into the file name. If that PNG is already in your run folder, it reuses the file instead of generating again. Re-run the cell: you only wait when one of those three changed.

Before you go on, look at the seed-7 image: does it show the scarf and the spectacles? If Sana skips a trait even here, don't count it in step 7.

In [ ]:
CHARACTER = (
    "a small red fox with a white chest, round brass spectacles and a green knitted scarf, "
    "holding a brass telescope; watercolor storybook illustration, soft painterly texture, no text"
)
scenes = {
    "snow": f"{CHARACTER}. The fox sits in the snow under a starry night sky.",
    "beach": f"{CHARACTER}. The fox sits on a sunny beach by the sea.",
}


def generate_image(prompt, seed=7, label="image", **settings):
    request = {"model": generator.model, "prompt": prompt, "seed": seed, **settings}
    path = folder / f"{label}-{image.request_id(**request)}.png"
    if not path.exists():
        generator.generate_image(prompt, out=path, seed=seed, **settings)
    return path


def compare_images(paths):
    display(Image(data=image_comparison(paths)))


images = [
    generate_image(scenes["snow"], seed=7, label="snow-seed-7"),
    generate_image(scenes["snow"], seed=8, label="snow-seed-8"),
]
compare_images(images)

## 7. Same character, new scene

Same character description, same seed, a different scene. Compare the two foxes: which traits survived, and which changed?

A prompt describes a character. It does not hold one fixed. Keeping a character across scenes takes more than a prompt and a seed. You will try this yourself in *Practice*, then write what you saw in *Your notes*.

In [ ]:
images = [
    generate_image(scenes["snow"], seed=7, label="snow-seed-7"),
    generate_image(scenes["beach"], seed=7, label="beach-seed-7"),
]
compare_images(images)

## Inspiration

Three more subjects. Each one repeats steps 6 and 7: two seeds in one scene, then the same seed in a new scene. Optional. Running a cell generates on your DevLab GPU.

### Astronaut cat

A character in a costume. Moon, then a sofa. Check whether the helmet and the backpack survive the move.

In [ ]:
CAT = (
    "a round orange tabby cat with a white muzzle, wearing a teal spacesuit, "
    "a clear round helmet and a small red backpack; "
    "playful 3D animated film illustration, soft lighting, no text"
)
cat_scenes = {
    "moon": f"{CAT}. The cat sits on a moon rock, with Earth in the dark sky.",
    "sofa": f"{CAT}. The cat sits on a cozy living-room sofa beside a sleeping dog.",
}


# Different seeds, same scene.
images = [
    generate_image(cat_scenes["moon"], seed=7, label="astronaut-cat-moon-seed-7"),
    generate_image(cat_scenes["moon"], seed=8, label="astronaut-cat-moon-seed-8"),
]
compare_images(images)

### Catalog mug

A product, not a character. Two seeds on white, then the same seed on white and pale blue. Check whether the handle and the lid stay the same mug.

In [ ]:
PRODUCT = (
    "one matte cream-colored reusable coffee mug with a cylindrical body, "
    "a large orange C-shaped handle and a dark charcoal lid; "
    "photorealistic product photography, front three-quarter view, "
    "soft studio lighting, subtle contact shadow, no logo, no text"
)
product_scenes = {
    "white": f"{PRODUCT}. The mug stands on a seamless white studio background.",
    "blue": f"{PRODUCT}. The mug stands on a seamless pale-blue studio background.",
}


# Different seeds: does the model invent different versions of the product?
images = [
    generate_image(product_scenes["white"], seed=9, label="catalog-mug-white-seed-9"),
    generate_image(product_scenes["white"], seed=8, label="catalog-mug-white-seed-8"),
]
compare_images(images)

### Christmas card

A holiday character and one gift. Workshop, then a doorstep. The prompt leaves the upper quarter empty, so a greeting could sit there.

In [ ]:
CHRISTMAS_DRAGON = (
    "a small mint-green dragon with short purple horns, "
    "wearing a red knitted scarf, holding one golden gift box "
    "tied with a wide red ribbon; "
    "charming watercolor Christmas card illustration, "
    "soft paper texture, warm gentle colors, no text"
)

christmas_scenes = {
    "workshop": (
        f"{CHRISTMAS_DRAGON}. The dragon sits at a wooden wrapping table "
        "beside a small Christmas tree with warm fairy lights. "
        "Simple uncluttered composition, upper quarter mostly empty ivory paper."
    ),
    "doorstep": (
        f"{CHRISTMAS_DRAGON}. The dragon stands outside a cozy cottage door "
        "in gently falling snow, warm light spilling from the window. "
        "Simple uncluttered composition, upper quarter mostly empty ivory paper."
    ),
}


# Choose your favorite interpretation.
images = [
    generate_image(christmas_scenes["workshop"], seed=7, label="christmas-card-workshop-seed-7"),
    generate_image(christmas_scenes["workshop"], seed=8, label="christmas-card-workshop-seed-8"),
]
compare_images(images)

## Practice

Your subject. Three scenes with the same character and seed. Change the quoted lines, then run. The quotes start as a fox, so a run with no edits still produces images.

Keep the text free of real people, brands and known characters.

### 1. One character across three scenes

Look at the first image before comparing the others. Count only the traits you can see. A trait the model skipped in scene 1 was never held fixed.

In [ ]:
# Change the quoted lines, then run.
TITLE = "Christmas Gift Cards factory"  # e.g. Add your title here
SUBJECT = "a small red fox with a white chest, round brass spectacles and a green knitted scarf"
STYLE = "watercolor storybook illustration, soft painterly texture, no text"

scenes = {
    "scene_1": f"{SUBJECT}. Sits in the snow under a starry night sky, holding a brass telescope",
    "scene_2": f"{SUBJECT}. Sitting in front of the laptop, and learning Media Generation course with DTC and Nebius.",
    "scene_3": f"{SUBJECT}. Decorating the home for christmas with Christmas lights and a Christmas tree.",
}

SEED = 7

my_images = [
    generate_image(f"{scenes[scene]} {STYLE}", SEED, label="mine", size="256x256")
    for scene in scenes
]
compare_images(my_images)

## A card to share

`share_card` builds one picture from the three Practice images. A line break in the headline is a line break on the card. The file is saved in your run folder. The QR code in the bottom-right corner opens the [course registration page](https://courses.datatalks.club/register/production-ai-agents-generative-media/).

In [ ]:
# A line break in the string is a line break on the card.
card = share_card(my_images, TITLE, out=folder / "share-card.png")
print(card)
display(Image(filename=str(card)))

## 8. Download results and delete the DevLab

Run the cell below to bundle your run folder (`first-image.png`, `metrics.json`, comparison images, and `share-card.png`) into a single `.zip` file. Right-click that `.zip` file in JupyterLab's left sidebar and click **Download**.

Then open [Devlabs in the Nebius web console](https://console.nebius.com/) and **Delete** the DevLab:

- **Stop** only stops GPU compute billing; the DevLab's container disk (250 GiB by default) continues to bill while stopped.
- **Delete** removes both the VM and the container disk so all billing stops.

You're done when:

- [ ] `runs/first-image.png` shows your image.
- [ ] Your run folder has the images from steps 4 to 7, Practice, `metrics.json`, and `share-card.png`.
- [ ] You've answered the four questions in *Your notes*.
- [ ] You've downloaded the `.zip` archive (and this notebook, if you edited *Your notes*) to your laptop.
- [ ] The DevLab is **deleted** in the console.

In [ ]:
if Path("runs/first-image.png").is_file():
    shutil.copy2("runs/first-image.png", folder / "first-image.png")

archive = shutil.make_archive(str(folder), "zip", root_dir=folder)
print(archive)
print("Then DELETE the DevLab in the Nebius console so disk billing stops.")